# PlantCLEF 2015 Upper-Bound Diagnostics

This notebook evaluates the `final_scnn_*` checkpoints saved by `plantclef_colab_upper_bound.ipynb` under `/content/drive/MyDrive/diploma_checkpoints/upper_bound`.

The upper-bound checkpoints are adapted on an official-test-derived split, so the metrics here are diagnostic and biased. Use them to inspect whether the target was reached, not as unbiased test results.

## 1. Runtime Check

Use a GPU runtime. The evaluation embeds reference and query images with both S-CNN stages.

In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Clone Or Update Project

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')

## 3. Mount Google Drive

In [ ]:
import os
import shutil
from pathlib import Path

if os.path.ismount('/content/drive') and Path('/content/drive/MyDrive').exists():
    print('Google Drive is already mounted at /content/drive')
else:
    if Path('/content/drive').exists() and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

## 4. Restore LeafScan Data And Rebuild Splits

This recreates the Paper60 training metadata and the same `adapt`/`holdout` split used by the upper-bound notebook.

In [ ]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma

ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

if [ ! -f "$ARCHIVE" ]; then
  echo "Missing LeafScan training archive: $ARCHIVE" >&2
  exit 2
fi
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing LeafScan test archive: $TEST_ARCHIVE" >&2
  exit 3
fi

rm -rf data/plantclef2015
mkdir -p data/plantclef2015

echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv

echo "extracting test archive: $TEST_ARCHIVE"
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

python - <<'PY2'
import csv
import random
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image

ROOT = Path('data/plantclef2015')
with (ROOT / 'leafscan_metadata.csv').open(newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
with (ROOT / 'test_leafscan_metadata.csv').open(newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))

test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    raise RuntimeError(f'Missing test species in train metadata: {missing_in_train}')

fieldnames = list(source_rows[0].keys())
train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    print('paper60 species with fewer than 6 train images:', underfilled_species)
    rows_by_species = defaultdict(list)
    for row in paper60_rows:
        rows_by_species[row['species']].append(row)
    augmented_dir = ROOT / 'leafscan' / 'augmented'
    augmented_dir.mkdir(parents=True, exist_ok=True)
    leafscan_root = ROOT / 'leafscan'
    angles = [180, 90, 270, 15, -15]
    augmented_rows = []
    for species in underfilled_species:
        species_rows = rows_by_species[species]
        if not species_rows:
            raise RuntimeError(f'Cannot augment {species}: no train rows found')
        needed = 6 - len(species_rows)
        for index in range(needed):
            base_row = species_rows[index % len(species_rows)]
            source_path = Path(base_row['image_path'])
            if not source_path.is_absolute():
                source_path = leafscan_root / source_path
            angle = angles[index % len(angles)]
            output_name = f"{source_path.stem}_aug_rot{angle}_{index + 1}.jpg".replace('-', 'm')
            output_path = augmented_dir / output_name
            with Image.open(source_path) as image:
                image.convert('RGB').rotate(angle, expand=True, fillcolor=(255, 255, 255)).save(output_path, quality=95)
            augmented_row = dict(base_row)
            augmented_row['image_path'] = str(output_path.relative_to(leafscan_root))
            if 'source_xml' in augmented_row:
                augmented_row['source_xml'] = f"{augmented_row['source_xml']}#aug_rot{angle}"
            augmented_rows.append(augmented_row)
    paper60_rows.extend(augmented_rows)
    print('paper60 augmented rows added:', len(augmented_rows))

with (ROOT / 'leafscan_paper60_metadata.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(paper60_rows)

split_fieldnames = list(test_rows[0].keys())
if 'split' not in split_fieldnames:
    split_fieldnames.append('split')
SEED = 20260526
ADAPT_RATIO = 0.60
rng = random.Random(SEED)
by_species = defaultdict(list)
for row in test_rows:
    by_species[row['species']].append(row)

split_rows = []
for species in sorted(by_species):
    items = sorted(by_species[species], key=lambda row: row['image_path'])
    rng.shuffle(items)
    n = len(items)
    if n == 1:
        adapt_count = 1
    else:
        adapt_count = max(1, int(round(n * ADAPT_RATIO)))
        adapt_count = min(adapt_count, n - 1)
    for index, row in enumerate(items):
        row = dict(row)
        row['split'] = 'adapt' if index < adapt_count else 'holdout'
        split_rows.append(row)

with (ROOT / 'test_leafscan_adapt_holdout_metadata.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=split_fieldnames)
    writer.writeheader()
    writer.writerows(sorted(split_rows, key=lambda row: row['image_path']))

print('leafscan source rows:', len(source_rows))
print('paper60 train rows:', len(paper60_rows))
print('paper60 train species:', len({row['species'] for row in paper60_rows}))
print('paper60 train genera:', len({row['genus'] for row in paper60_rows}))
print('official test rows:', len(test_rows))
print('official test species:', len(test_species))
print('official test genera:', len({row['genus'] for row in test_rows}))
print('upper-bound split counts:', dict(Counter(row['split'] for row in split_rows)))
print('upper-bound holdout species:', len({row['species'] for row in split_rows if row['split'] == 'holdout'}))
PY2

## 5. Locate Upper-Bound Checkpoints

In [ ]:
from pathlib import Path
import re
import shutil
import pandas as pd

DRIVE_DIR = Path('/content/drive/MyDrive/diploma_checkpoints/upper_bound')
LOCAL_CHECKPOINT_DIR = Path('/content/diploma/checkpoints/upper_bound_eval')
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_DIR.exists():
    raise FileNotFoundError(f'Upper-bound checkpoint directory does not exist: {DRIVE_DIR}')

candidates = sorted(DRIVE_DIR.glob('final_scnn_*.pt'), key=lambda path: path.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f'No final_scnn_*.pt checkpoints found in {DRIVE_DIR}')

rows = []
for source in candidates:
    target = LOCAL_CHECKPOINT_DIR / source.name
    shutil.copy2(source, target)
    rows.append({
        'name': source.name,
        'drive_path': str(source),
        'local_path': str(target),
        'size_mb': round(source.stat().st_size / 1024 / 1024, 1),
        'mtime': pd.to_datetime(source.stat().st_mtime, unit='s'),
    })
ckpt_df = pd.DataFrame(rows).sort_values('name')
display(ckpt_df)

round_re = re.compile(r'^final_scnn_(genus|species)_vgg16(?:_round(\d+))?(_best)?\.pt$')
by_key = {}
for row in rows:
    match = round_re.match(row['name'])
    if not match:
        continue
    stage, round_id, best = match.groups()
    key = 'final' if round_id is None else f'round{round_id}'
    variant = 'best' if best else 'final'
    by_key.setdefault(key, {}).setdefault(stage, {})[variant] = Path(row['local_path'])

pairs = []
for key, stages in sorted(by_key.items(), key=lambda item: (item[0] != 'final', item[0])):
    if 'genus' not in stages or 'species' not in stages:
        continue
    for variant in ['final', 'best']:
        genus = stages['genus'].get(variant) or stages['genus'].get('final') or stages['genus'].get('best')
        species = stages['species'].get(variant) or stages['species'].get('final') or stages['species'].get('best')
        if genus and species:
            pairs.append({'label': f'{key}_{variant}', 'genus_checkpoint': genus, 'species_checkpoint': species})

if not pairs:
    raise RuntimeError('No complete genus/species checkpoint pairs found.')

print('checkpoint pairs:')
for pair in pairs:
    print(pair['label'], pair['genus_checkpoint'].name, pair['species_checkpoint'].name)

## 6. Evaluation Helpers

In [ ]:
import copy
import inspect
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import yaml

PROJECT_DIR = Path('/content/diploma')
src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from plant_classifier.data import filter_records_by_split, limit_records_by_species, load_metadata_csv
from plant_classifier.inference.scnn import TwoStageSiamesePredictor
from plant_classifier.models.siamese import BackboneSpec, build_siamese_network
from plant_classifier.training.genus_eval import evaluate_genus_retrieval, select_genus_references
from plant_classifier.training.species_eval import build_reference_embeddings, evaluate_species_retrieval, select_species_references
from plant_classifier.training.validation import validate_records_exist

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_CONFIG_PATH = PROJECT_DIR / 'configs/leafscan_paper60_training.yaml'
TEST_CONFIG_PATH = PROJECT_DIR / 'configs/leafscan_test.yaml'
with TRAIN_CONFIG_PATH.open('r', encoding='utf-8') as file:
    BASE_CONFIG = yaml.safe_load(file)
with TEST_CONFIG_PATH.open('r', encoding='utf-8') as file:
    TEST_CONFIG = yaml.safe_load(file)

UPPER_CONFIG = copy.deepcopy(BASE_CONFIG)
UPPER_CONFIG['dataset'] = {
    'name': 'PlantCLEF2015LeafScanOfficialTestAdaptHoldout',
    'root': 'data/plantclef2015/test_leafscan/leafscan',
    'metadata': 'data/plantclef2015/test_leafscan_adapt_holdout_metadata.csv',
    'image_column': 'image_path',
    'family_column': 'family',
    'genus_column': 'genus',
    'species_column': 'species',
}

IMAGE_SIZE = int(BASE_CONFIG['views']['global']['image_size'])
CROP_SIZE = int(BASE_CONFIG['views']['local']['crop_size'])
CROP_POSITION = str(BASE_CONFIG['views']['local'].get('crop_position', 'center'))
PREPROCESSING = bool(BASE_CONFIG.get('preprocessing', {}).get('enabled', False))
MODEL_SPEC = BackboneSpec(name=BASE_CONFIG['model']['backbone'], pretrained=False)


def load_records_from_dataset(dataset: dict):
    return load_metadata_csv(
        metadata_path=Path(dataset['metadata']),
        dataset_root=Path(dataset['root']),
        image_column=dataset['image_column'],
        family_column=dataset['family_column'],
        genus_column=dataset['genus_column'],
        species_column=dataset['species_column'],
    )


def apply_subset(records, dataset_config):
    subset = dataset_config.get('subset')
    if not subset:
        return records
    return limit_records_by_species(
        records,
        max_species=subset.get('max_species'),
        min_images_per_species=int(subset.get('min_images_per_species', 1)),
        max_images_per_species=subset.get('max_images_per_species'),
        seed=subset.get('seed'),
    )


def load_siamese(checkpoint_path: Path):
    model = build_siamese_network(MODEL_SPEC).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    return model.eval()


def build_reference_embeddings_compat(**kwargs):
    params = inspect.signature(build_reference_embeddings).parameters
    supported = {key: value for key, value in kwargs.items() if key in params}
    return build_reference_embeddings(**supported)


def build_two_stage_predictor_compat(**kwargs):
    params = inspect.signature(TwoStageSiamesePredictor.__init__).parameters
    supported = {key: value for key, value in kwargs.items() if key in params}
    dropped = sorted(set(kwargs) - set(supported))
    if dropped:
        print('TwoStageSiamesePredictor does not support:', ', '.join(dropped))
    return TwoStageSiamesePredictor(**supported)


@torch.inference_mode()
def evaluate_pair(label, genus_checkpoint, species_checkpoint, reference_records, query_records, *, reference_seed=42, note=''):
    validate_records_exist(reference_records)
    validate_records_exist(query_records)
    genus_model = load_siamese(Path(genus_checkpoint))
    species_model = load_siamese(Path(species_checkpoint))
    allowed_species = {record.species for record in query_records}
    selected_reference_records = [record for record in reference_records if record.species in allowed_species]
    genus_references = select_genus_references(selected_reference_records, references_per_genus=6, seed=reference_seed)
    species_references = select_species_references(reference_records, references_per_species=6, allowed_species=allowed_species)
    validate_records_exist(genus_references)
    validate_records_exist(species_references)

    genus_result = evaluate_genus_retrieval(
        model=genus_model,
        references=genus_references,
        queries=query_records,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        top_ks=(5, 15, 30),
        device=DEVICE,
        preprocessing=PREPROCESSING,
        score_mode='l1',
    )
    genus_embeddings = build_reference_embeddings_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=genus_references,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        device=DEVICE,
    )
    species_embeddings = build_reference_embeddings_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=species_references,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        device=DEVICE,
    )
    predictor = build_two_stage_predictor_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=species_embeddings,
        genus_references=genus_embeddings,
        genus_candidates=30,
        top_k=5,
        image_size=IMAGE_SIZE,
        local_crop_size=CROP_SIZE,
        local_crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        genus_score_mode='l1',
        species_score_mode='l1',
        species_aggregation='max',
        genus_candidate_mode='unique',
        genus_weight_mode='score',
        device=str(DEVICE),
    )
    species_result = evaluate_species_retrieval(
        predictor=predictor,
        queries=query_records,
        top_ks=(1, 3, 5),
        ranking_limit=len({record.species for record in species_references}),
    )
    return {
        'label': label,
        'note': note,
        'genus_checkpoint': Path(genus_checkpoint).name,
        'species_checkpoint': Path(species_checkpoint).name,
        'references_genus': genus_result.references,
        'references_species': species_result.references,
        'queries': species_result.queries,
        'top5_genus': genus_result.accuracies[5],
        'top15_genus': genus_result.accuracies[15],
        'top30_genus': genus_result.accuracies[30],
        'top1_species': species_result.accuracies[1],
        'top3_species': species_result.accuracies[3],
        'top5_species': species_result.accuracies[5],
        'plantclef_s': species_result.plantclef_s,
    }

print('device:', DEVICE)

## 7. Evaluate Upper-Bound Holdout Split

References are taken from the `adapt` split and queries from the `holdout` split. This is the target diagnostic protocol from the upper-bound notebook.

In [ ]:
all_upper_records = load_records_from_dataset(UPPER_CONFIG['dataset'])
adapt_records = filter_records_by_split(all_upper_records, 'adapt')
holdout_records = filter_records_by_split(all_upper_records, 'holdout')
print('adapt:', len(adapt_records), 'holdout:', len(holdout_records))
print('holdout species:', len({record.species for record in holdout_records}))
print('holdout genera:', len({record.genus for record in holdout_records}))

holdout_rows = []
for pair in pairs:
    print()
    print('=== evaluating holdout', pair['label'], '===')
    row = evaluate_pair(
        pair['label'],
        pair['genus_checkpoint'],
        pair['species_checkpoint'],
        adapt_records,
        holdout_records,
        reference_seed=42,
        note='adapt->holdout',
    )
    holdout_rows.append(row)
    print(row)

holdout_df = pd.DataFrame(holdout_rows).sort_values(['top1_species', 'top30_genus', 'plantclef_s'], ascending=False)
display(holdout_df)
OUT_DIR = Path('/content/drive/MyDrive/diploma_diagnostics/upper_bound_final_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)
holdout_csv = OUT_DIR / 'upper_bound_holdout_results.csv'
holdout_df.to_csv(holdout_csv, index=False)
print('saved:', holdout_csv)

## 8. Evaluate Full Official Test Diagnostic

This uses Paper60 train references and all official test images as queries. Because the evaluated checkpoints may have been adapted on test-derived images, this is a biased diagnostic only.

In [ ]:
train_records = apply_subset(load_records_from_dataset(BASE_CONFIG['dataset']), BASE_CONFIG['dataset'])
test_records = load_records_from_dataset(TEST_CONFIG['dataset'])
print('paper60 six-shot train references pool:', len(train_records))
print('official test queries:', len(test_records))

full_rows = []
for pair in pairs:
    print()
    print('=== evaluating full test', pair['label'], '===')
    row = evaluate_pair(
        pair['label'],
        pair['genus_checkpoint'],
        pair['species_checkpoint'],
        train_records,
        test_records,
        reference_seed=42,
        note='paper60_train->official_test_biased',
    )
    full_rows.append(row)
    print(row)

full_df = pd.DataFrame(full_rows).sort_values(['top1_species', 'top30_genus', 'plantclef_s'], ascending=False)
display(full_df)
OUT_DIR = Path('/content/drive/MyDrive/diploma_diagnostics/upper_bound_final_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)
full_csv = OUT_DIR / 'upper_bound_full_test_biased_results.csv'
full_df.to_csv(full_csv, index=False)
print('saved:', full_csv)

## 9. Plot Summary

In [ ]:
import matplotlib.pyplot as plt

OUT_DIR = Path('/content/drive/MyDrive/diploma_diagnostics/upper_bound_final_eval')
for title, df, filename in [
    ('Upper-bound holdout', holdout_df, 'upper_bound_holdout_summary.png'),
    ('Full official test diagnostic', full_df, 'upper_bound_full_test_biased_summary.png'),
]:
    if df.empty:
        continue
    plot_df = df.sort_values('label')
    fig, ax = plt.subplots(figsize=(max(8, len(plot_df) * 1.2), 4.8))
    x = range(len(plot_df))
    ax.bar([v - 0.2 for v in x], plot_df['top1_species'], width=0.2, label='Top-1 species')
    ax.bar(x, plot_df['top5_species'], width=0.2, label='Top-5 species')
    ax.bar([v + 0.2 for v in x], plot_df['top30_genus'], width=0.2, label='Top-30 genus')
    ax.set_xticks(list(x))
    ax.set_xticklabels(plot_df['label'], rotation=35, ha='right')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('accuracy')
    ax.set_title(title)
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend()
    fig.tight_layout()
    path = OUT_DIR / filename
    fig.savefig(path, dpi=180)
    display(fig)
    plt.close(fig)
    print('saved:', path)

## 10. Inspect Saved Upper-Bound Logs

In [ ]:
from pathlib import Path

log_files = sorted(DRIVE_DIR.glob('diploma-upper_*.log'), key=lambda path: path.stat().st_mtime, reverse=True)
print('latest logs:')
for path in log_files[:10]:
    print(path.name, path.stat().st_size, 'bytes')

if log_files:
    print()
    print('--- tail of latest log ---')
    text = log_files[0].read_text(errors='replace')
    print(chr(10).join(text.splitlines()[-120:]))

## 11. Build Final Reference Index Artifact

The index is not a third trainable model. It is a saved reference-embedding table. This cell builds **only** the app-ready index for `final_scnn_genus_vgg16.pt` + `final_scnn_species_vgg16.pt`. It uses the same `adapt` references as the final-checkpoint evaluation and does not use round or best checkpoints.


In [12]:
from pathlib import Path

from plant_classifier.inference.scnn import save_reference_index

OUT_DIR = Path('/content/drive/MyDrive/diploma_checkpoints/upper_bound')
OUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_GENUS = OUT_DIR / 'final_scnn_genus_vgg16.pt'
FINAL_SPECIES = OUT_DIR / 'final_scnn_species_vgg16.pt'
missing = [path for path in (FINAL_GENUS, FINAL_SPECIES) if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Final checkpoints are not available yet. Missing: '
        + ', '.join(str(path) for path in missing)
        + '. Run/finish the upper-bound notebook until it saves final_scnn_genus_vgg16.pt '
        + 'and final_scnn_species_vgg16.pt.'
    )

LOCAL_FINAL_GENUS = LOCAL_CHECKPOINT_DIR / FINAL_GENUS.name
LOCAL_FINAL_SPECIES = LOCAL_CHECKPOINT_DIR / FINAL_SPECIES.name
shutil.copy2(FINAL_GENUS, LOCAL_FINAL_GENUS)
shutil.copy2(FINAL_SPECIES, LOCAL_FINAL_SPECIES)
print('final genus:', FINAL_GENUS, f'({FINAL_GENUS.stat().st_size / 1024 / 1024:.1f} MB)')
print('final species:', FINAL_SPECIES, f'({FINAL_SPECIES.stat().st_size / 1024 / 1024:.1f} MB)')


@torch.inference_mode()
def build_final_index_artifact(reference_records, output_path: Path):
    genus_model = load_siamese(LOCAL_FINAL_GENUS)
    species_model = load_siamese(LOCAL_FINAL_SPECIES)
    genus_references = select_genus_references(reference_records, references_per_genus=6, seed=42)
    species_references = select_species_references(reference_records, references_per_species=6)
    genus_embeddings = build_reference_embeddings_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=genus_references,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        device=DEVICE,
    )
    species_embeddings = build_reference_embeddings_compat(
        genus_model=genus_model,
        species_model=species_model,
        references=species_references,
        image_size=IMAGE_SIZE,
        crop_size=CROP_SIZE,
        crop_position=CROP_POSITION,
        preprocessing=PREPROCESSING,
        device=DEVICE,
    )
    save_reference_index(species_embeddings, output_path, genus_references=genus_embeddings)
    print(
        f'saved {len(genus_embeddings)} genus and '
        f'{len(species_embeddings)} species embeddings -> {output_path} '
        f'({output_path.stat().st_size / 1024 / 1024:.1f} MB)'
    )


if 'adapt_records' not in globals():
    all_upper_records = load_records_from_dataset(UPPER_CONFIG['dataset'])
    adapt_records = filter_records_by_split(all_upper_records, 'adapt')

FINAL_INDEX = OUT_DIR / 'final_reference_index_adapt_vgg16.pt'
build_final_index_artifact(adapt_records, FINAL_INDEX)

print('Use this as app reference_index:')
print(FINAL_INDEX)

final genus: /content/drive/MyDrive/diploma_checkpoints/upper_bound/final_scnn_genus_vgg16.pt (512.2 MB)
final species: /content/drive/MyDrive/diploma_checkpoints/upper_bound/final_scnn_species_vgg16.pt (512.2 MB)
saved 109 genus and 124 species embeddings -> /content/drive/MyDrive/diploma_checkpoints/upper_bound/final_reference_index_adapt_vgg16.pt
Use this as app reference_index:
/content/drive/MyDrive/diploma_checkpoints/upper_bound/final_reference_index_adapt_vgg16.pt
